In [36]:
import sys
from pathlib import Path
import pandas as pd 
import numpy as np

sys.path.append(str(Path.cwd().parent))

from src.data_loading import load_all_stations
from src.cleaning import replace_missing_values

raw = load_all_stations()
cleaned = replace_missing_values(raw)
print('Raw shape:', raw.shape)
print('Columns:', raw.columns.to_list())
print(raw.head(3))


Raw shape: (61130, 28)
Columns: ['STATION', 'DATE', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'TEMP', 'TEMP_ATTRIBUTES', 'DEWP', 'DEWP_ATTRIBUTES', 'SLP', 'SLP_ATTRIBUTES', 'STP', 'STP_ATTRIBUTES', 'VISIB', 'VISIB_ATTRIBUTES', 'WDSP', 'WDSP_ATTRIBUTES', 'MXSPD', 'GUST', 'MAX', 'MAX_ATTRIBUTES', 'MIN', 'MIN_ATTRIBUTES', 'PRCP', 'PRCP_ATTRIBUTES', 'SNDP', 'FRSHTT']
       STATION       DATE   LATITUDE  LONGITUDE  ELEVATION  \
0  41184099999 1996-01-01  25.613483  55.938817      31.08   
1  41184099999 1996-01-02  25.613483  55.938817      31.08   
2  41184099999 1996-01-03  25.613483  55.938817      31.08   

                               NAME  TEMP  TEMP_ATTRIBUTES  DEWP  \
0  RAS AL KHAIMAH INTERNATIONAL, AE  68.8               22  58.7   
1  RAS AL KHAIMAH INTERNATIONAL, AE  70.4               21  60.4   
2  RAS AL KHAIMAH INTERNATIONAL, AE  70.0               23  59.5   

   DEWP_ATTRIBUTES  ...  MXSPD   GUST   MAX  MAX_ATTRIBUTES   MIN  \
0               21  ...    7.0  999.9  

61,130 daily records across six stations. Raw shape before adding derived notebook columns: (61130, 28).

In [38]:
print(f"Rows per station:\n{raw['NAME'].value_counts()}")
print(f"Earliest date for each station:\n{raw.groupby('NAME')['DATE'].min()}")
print(f"Latest date for each station:\n{raw.groupby('NAME')['DATE'].max()}")

Rows per station:
NAME
ABU DHABI INTERNATIONAL, AE         10219
DUBAI INTERNATIONAL, AE             10216
SHARJAH INTERNATIONAL, AE           10205
AL AIN INTERNATIONAL, AE            10192
FUJAIRAH INTERNATIONAL, AE          10161
RAS AL KHAIMAH INTERNATIONAL, AE    10137
Name: count, dtype: int64
Earliest date for each station:
NAME
ABU DHABI INTERNATIONAL, AE        1996-01-01
AL AIN INTERNATIONAL, AE           1996-01-01
DUBAI INTERNATIONAL, AE            1996-01-01
FUJAIRAH INTERNATIONAL, AE         1996-01-01
RAS AL KHAIMAH INTERNATIONAL, AE   1996-01-01
SHARJAH INTERNATIONAL, AE          1996-01-01
Name: DATE, dtype: datetime64[us]
Latest date for each station:
NAME
ABU DHABI INTERNATIONAL, AE        2023-12-31
AL AIN INTERNATIONAL, AE           2023-12-31
DUBAI INTERNATIONAL, AE            2023-12-31
FUJAIRAH INTERNATIONAL, AE         2023-12-31
RAS AL KHAIMAH INTERNATIONAL, AE   2023-12-31
SHARJAH INTERNATIONAL, AE          2023-12-31
Name: DATE, dtype: datetime64[us]


In [4]:
print(f"Number of days between earliest and latest date for each station:\n{raw.groupby('NAME')['DATE'].apply(lambda x: (x.max() - x.min()).days)}")

Number of days between earliest and latest date for each station:
NAME
ABU DHABI INTERNATIONAL, AE         10226
AL AIN INTERNATIONAL, AE            10226
DUBAI INTERNATIONAL, AE             10226
FUJAIRAH INTERNATIONAL, AE          10226
RAS AL KHAIMAH INTERNATIONAL, AE    10226
SHARJAH INTERNATIONAL, AE           10226
Name: DATE, dtype: int64


In [44]:
calendar = pd.date_range('1996-01-01', '2023-12-31', freq='D')
coverage = raw.groupby('NAME')['DATE'].apply(lambda dates: len(calendar.difference(pd.DatetimeIndex(dates))))
print('Missing calendar days per station:')
print(coverage)
print('Total missing days:', coverage.sum())
print('Expected rows:', len(calendar) * raw['NAME'].nunique())

Missing calendar days per station:
NAME
ABU DHABI INTERNATIONAL, AE          8
AL AIN INTERNATIONAL, AE            35
DUBAI INTERNATIONAL, AE             11
FUJAIRAH INTERNATIONAL, AE          66
RAS AL KHAIMAH INTERNATIONAL, AE    90
SHARJAH INTERNATIONAL, AE           22
Name: DATE, dtype: int64
Total missing days: 232
Expected rows: 61362


Inspecting Ras Al Khaimah gaps

In [51]:
dates = pd.date_range('1996-01-01', '2023-12-31', freq='D')
rak = raw[raw['STATION'] == 41184099999]
present = pd.DatetimeIndex(rak["DATE"])

missing_dates = dates.difference(present)
print(f"missing days from rak: {len(missing_dates)}")

missing_by_month = pd.Series(missing_dates).dt.to_period("M").value_counts().sort_index()
print(f"Missing by Month: \n{missing_by_month}")


missing days from rak: 10227
Missing by Month: 
1996-01    31
1996-02    29
1996-03    31
1996-04    30
1996-05    31
           ..
2023-08    31
2023-09    30
2023-10    31
2023-11    30
2023-12    31
Freq: M, Name: count, Length: 336, dtype: int64


Checking Monthly precipitation completeness (D3)

In [62]:
monthly = cleaned.assign(MONTH=cleaned['DATE'].dt.to_period('M'))
months = pd.period_range('1996-01', '2023-12', freq='M')
full_index = pd.MultiIndex.from_product(
    [sorted(monthly['NAME'].unique()), months], names=['NAME', 'MONTH']
)
valid_days = monthly.groupby(['NAME', 'MONTH'])['PRCP'].count().reindex(full_index, fill_value=0)
print('Station-months:', len(valid_days))
for threshold in [28, 25, 20]:
    print(f'Below {threshold} valid days:', (valid_days < threshold).sum())
print('Completely absent months:', (valid_days == 0).sum())

Station-months: 2016
Below 28 valid days: 127
Below 25 valid days: 24
Below 20 valid days: 4
Completely absent months: 1


Decision D3 : Require at least 28 valid PRCP days to retain a monthly rainfall target.

In [64]:
print(raw[['STATION', 'FRSHTT']].dtypes)
print(raw['FRSHTT'].head())

STATION    str
FRSHTT     str
dtype: object
0    000000
1    010000
2    000000
3    010000
4    000000
Name: FRSHTT, dtype: str


Overall missingness after cleaning

In [65]:
missing_pct = (cleaned.isna().mean() * 100).sort_values(ascending=False)
print(missing_pct.to_string())

GUST                93.214461
SLP                 17.147064
PRCP                 1.349583
MXSPD                0.171765
VISIB                0.104695
MAX                  0.004908
MIN                  0.004908
DEWP                 0.001636
LONGITUDE            0.000000
LATITUDE             0.000000
STATION              0.000000
DATE                 0.000000
SLP_ATTRIBUTES       0.000000
DEWP_ATTRIBUTES      0.000000
TEMP                 0.000000
TEMP_ATTRIBUTES      0.000000
NAME                 0.000000
ELEVATION            0.000000
VISIB_ATTRIBUTES     0.000000
STP                  0.000000
WDSP                 0.000000
WDSP_ATTRIBUTES      0.000000
STP_ATTRIBUTES       0.000000
MAX_ATTRIBUTES       0.000000
MIN_ATTRIBUTES       0.000000
PRCP_ATTRIBUTES      0.000000
SNDP                 0.000000
FRSHTT               0.000000


In [69]:
slp_by_station = cleaned.groupby('NAME')['SLP'].apply(lambda x: x.isna().mean() * 100)
print('SLP missing percentage by station:')
print(slp_by_station.sort_values(ascending=False).to_string())

slp_by_year = cleaned.assign(YEAR=cleaned['DATE'].dt.year).groupby(['NAME', 'YEAR'])['SLP'].apply(
    lambda x: x.isna().mean() * 100
)
print('Station-years with 100% missing SLP:')
print(slp_by_year[slp_by_year == 100].to_string())

SLP missing percentage by station:
NAME
FUJAIRAH INTERNATIONAL, AE          44.631434
AL AIN INTERNATIONAL, AE            22.556907
DUBAI INTERNATIONAL, AE             12.852388
RAS AL KHAIMAH INTERNATIONAL, AE     8.197692
SHARJAH INTERNATIONAL, AE            8.123469
ABU DHABI INTERNATIONAL, AE          6.605343
Station-years with 100% missing SLP:
NAME                        YEAR
AL AIN INTERNATIONAL, AE    2001    100.0
                            2002    100.0
                            2003    100.0
FUJAIRAH INTERNATIONAL, AE  1996    100.0
                            1997    100.0
                            1998    100.0
                            1999    100.0
                            2000    100.0
                            2001    100.0
                            2002    100.0
                            2003    100.0
                            2022    100.0


Decision D6: Exclude SLP from the main feature set because long, station-dependent gaps would require extensive imputation and complicate cross-station comparisons.